# Pretraining Dataset Groups

This notebook groups datasets the [GIFT-Eval](https://github.com/SalesforceAIResearch/gift-eval) pretraining split based on the models proposed in the [Ensemble TEMPO slides](https://docs.google.com/presentation/d/1GxL_qUQKizv5C_RPzxYiJjxJSPD6-81x1VPOxxYXAl0/edit?slide=id.g35d6bf22e47_0_10#slide=id.g35d6bf22e47_0_10).

Ensemble TEMPO will have three types of models:
- **General** 
  - Three models across three forecasting terms (short, medium, and long)
- **Domain-specific** 
  - 15 models across seven domains and three forecasting terms
- **Dataset-specfific** 
  - 97+ models across each dataset

## Data

### Loading

Load the pretraining split's metadata.

In [57]:
import pandas as pd
from pathlib import Path

split = "pretrain"
metadata_path = Path("resources") / split / "metadata.csv"

df = pd.read_csv(metadata_path)
df.head()

,name,term,freq,domain,num_entries,target_dim,_min_series_length,sum_series_length,prediction_length,windows
0,bull,short,H,Energy,41.0,1,17544,719304,48,20
1,bull,medium,H,Energy,41.0,1,17544,719304,480,4
2,bull,long,H,Energy,41.0,1,17544,719304,720,3
3,cmip6_1885,short,6H,Climate,434176.0,53,7300,3169484800,48,16
4,cmip6_1885,medium,6H,Climate,434176.0,53,7300,3169484800,480,2


### EDA

View all of the unique forecasting terms and count the number of datasets that belong to each term. 

Each forecasting term multiplies the original prediciton length by a given multipler:
| Term    | Multipler   
|:--------|:--------|
| Short   | 1       |
| Medium  | 10      |
| Long    | 15      |

In [ ]:
def get_count_df(
    df: pd.DataFrame,
    columns: list[str],
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Counts the number of unique combinations of all unique values in the
    specified columns of the given a DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        columns (list[str]): A list of column names to group by.
        ascending (bool, optional): If True, sort the result in ascending order
            of count. Defaults to False (descending order).

    Returns:
        pd.DataFrame: A DataFrame with the grouped columns and a 'count'
            column, sorted by count.
    """
    count_df = df.groupby(columns).size().reset_index(name="count")
    return count_df.sort_values(by="count", ascending=ascending).reset_index(drop=True)


term_counts = get_count_df(df, columns=["term"], ascending=False)

print(f"Number of unique terms: {len(term_counts)}")
display(term_counts)

Number of unique terms: 3


,term,count
0,long,152
1,medium,152
2,short,152


View all of the unique domains and count the number of datasets that belong to each domain.

In [59]:
domain_counts = get_count_df(df, columns=["domain"], ascending=False)

print(f"Number of unique domains: {len(domain_counts)}")
display(domain_counts)

Number of unique domains: 9


,domain,count
0,Climate,201
1,Energy,84
2,Transport,75
3,Econ/Fin,51
4,CloudOps,9
5,Healthcare,9
6,Nature,9
7,Sales,9
8,Web,9


View all of the unique domain-term combinations.

In [60]:
domain_term_counts = get_count_df(
    df,
    columns=["domain", "term"],
    ascending=False,
)

# Convert each forecasting term into its own column
pivoted_df = domain_term_counts.pivot(
    index="domain",
    columns="term",
    values="count",
).reset_index()

# Remove old "term" column
pivoted_df.columns.name = None

# Reorder columns
pivoted_df = pivoted_df[
    [
        "domain",
        "short",
        "medium",
        "long",
    ]
]

print(f"Number of unique domain-term combinations: {len(domain_term_counts)}")
display(pivoted_df)

Number of unique domain-term combinations: 27


,domain,short,medium,long
0,Climate,67,67,67
1,CloudOps,3,3,3
2,Econ/Fin,17,17,17
3,Energy,28,28,28
4,Healthcare,3,3,3
5,Nature,3,3,3
6,Sales,3,3,3
7,Transport,25,25,25
8,Web,3,3,3


View some of the dataset name-term combinations. 
- **Note:** for the sake of brevity, we'll only display a few of the dataset name-term combinations

In [61]:
name_term_combinations = df[["name", "term"]]

print(f"Number of unique name-term pairs: {len(name_term_combinations)}")
name_term_combinations.head()

Number of unique name-term pairs: 456


,name,term
0,bull,short
1,bull,medium
2,bull,long
3,cmip6_1885,short
4,cmip6_1885,medium
